# Session 43: Feature Engineering and Explaining Models

Data Science with Python, Level 3, Stage 6: Applied ML

## Learning Objectives

By the end of this session you will be able to:

1. Create features from dates.
2. Encode cycles like months with sine and cosine.
3. Build features from domain knowledge.
4. Measure feature value with cross-validation.
5. Explain models with permutation importance.
6. Read partial dependence and what-if plots.

## Where We Are

Session 42 showed that tuning gives modest gains. The bigger lever is usually **feature engineering**: turning raw columns into inputs that make the pattern easy to learn. A date is just text to a model until you extract the year, the month and the weekday.

The second half of today answers the question every client asks: **why** does the model predict this? You will use two model-agnostic tools, permutation importance and partial dependence, which work for any model, from linear regression to boosting. We use the 731 days of bike rentals.

## Setup

`cv_r2` is a small helper that returns the shuffled 5-fold R2 for a model and a list of columns. `linear` and `boost` are the two models we compare.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import r2_score, mean_absolute_error

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
bikes = pd.read_csv(DATA_URL + "bike_rentals.csv")
bikes["date"] = pd.to_datetime(bikes["date"])
folds = KFold(n_splits=5, shuffle=True, random_state=0)

linear = LinearRegression()
boost = HistGradientBoostingRegressor(random_state=0)

def cv_r2(model, cols):
    return cross_val_score(model, bikes[cols].astype(float), bikes["rentals"], cv=folds).mean().round(3)

## 1. Features from Dates

In [ ]:
base = ["temp_c", "humidity", "wind_kmh", "working_day", "holiday"]
print("Linear:  ", cv_r2(linear, base))
print("Boosting:", cv_r2(boost, base))

In [ ]:
print(bikes.groupby(bikes["date"].dt.year)["rentals"].mean().round())

In [ ]:
bikes["year"] = bikes["date"].dt.year - 2023
bikes["month"] = bikes["date"].dt.month
bikes["day_of_week"] = bikes["date"].dt.dayofweek
bikes["weekend"] = (bikes["day_of_week"] >= 5).astype(int)

with_year = base + ["year"]
print("Linear:  ", cv_r2(linear, with_year))
print("Boosting:", cv_r2(boost, with_year))

In [ ]:
day = bikes["date"].dt.dayofyear
bikes["doy_sin"] = np.sin(2 * np.pi * day / 365)
bikes["doy_cos"] = np.cos(2 * np.pi * day / 365)

plt.scatter(bikes["doy_sin"], bikes["doy_cos"],
            c=bikes["month"], cmap="twilight", s=8)
plt.colorbar(label="Month")
plt.title("Each day is a point on a circle")
plt.show()

## 2. Features from Domain Knowledge

In [ ]:
print(bikes.groupby("weather")["rentals"].mean().round())

bikes["bad_weather"] = (bikes["weather"] == "Rain").astype(int)
bikes["temp_sq"] = bikes["temp_c"] ** 2

In [ ]:
features = base + ["year", "doy_sin", "doy_cos", "temp_sq", "bad_weather", "weekend"]
print("Linear:  ", cv_r2(linear, features))
print("Boosting:", cv_r2(boost, features))

| Recipe | Example | Code idea |
| --- | --- | --- |
| Date parts | year, month, weekday | df["date"].dt.month |
| Cycles | day of year, hour | np.sin(2 * np.pi * x / period) |
| Flags | rain, weekend, new customer | (df["x"] == value).astype(int) |
| Ratios | price per square metre | df["price"] / df["area"] |
| Bends | temperature squared | df["temp"] ** 2 |
| Group statistics | average sales of the store | inside a pipeline, learned in fit |

Group statistics that use the target must be learned inside the pipeline, as in the MeanEncoder exercise of Session 41.

## 3. Explaining the Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    bikes[features].astype(float), bikes["rentals"], test_size=0.25, random_state=0)
model = HistGradientBoostingRegressor(random_state=0).fit(X_train, y_train)

pred = model.predict(X_test)
print("Test R2:", round(r2_score(y_test, pred), 3))
print("Test MAE:", round(mean_absolute_error(y_test, pred)))

In [ ]:
result = permutation_importance(model, X_test, y_test,
                                n_repeats=10, random_state=0)
drop = pd.Series(result.importances_mean, index=features)
drop.sort_values().plot.barh(color="tab:purple")
plt.title("Drop in R2 when shuffled")
plt.show()

In [ ]:
PartialDependenceDisplay.from_estimator(model, X_train, ["temp_c", "humidity"])
plt.show()

In [ ]:
day = X_test.iloc[[0]].copy()
temps = [5, 10, 15, 20, 25]
what_if = pd.concat([day.assign(temp_c=t, temp_sq=t ** 2) for t in temps])
print(pd.Series(model.predict(what_if).round(), index=temps))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Date as a number | Feeding the raw date into a model | Extract year, month, weekday and cycles. |
| Leaky features | Using next week's rentals as a feature | Only use information known at prediction time. |
| Importance as cause | Calling an important feature a cause | Importance shows reliance, not causation. |
| Correlated twins | Trusting low importance of a twin feature | Shuffling one twin hides it behind the other. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Weekday Pattern

Show the average rentals for each `day_of_week` (0 is Monday) and for working days versus non-working days. Is the weekend effect large?

In [ ]:
# Write your code below

### In-Class Exercise 2: Month as Category

Compare three ways to add the month to the linear model: as a number 1 to 12, as one-hot columns (`pd.get_dummies(bikes["month"], prefix="m")`), and as sine and cosine. Use `cv_r2`-style cross-validation for each.

In [ ]:
# Write your code below

### In-Class Exercise 3: Explain the Linear Model

Fit a linear regression on all engineered features and compute permutation importance on the test set. Compare the ranking with the boosting model's ranking.

In [ ]:
# Write your code below

### In-Class Exercise 4: Rain and Temperature Together

Draw a two-feature partial dependence plot for `temp_c` and `bad_weather` together (pass `[("temp_c", "bad_weather")]` to `PartialDependenceDisplay.from_estimator`). What does it show?

In [ ]:
# Write your code below

## Summary

- Feature engineering often beats tuning.
- Extract year, month and weekday with the `.dt` accessor.
- Encode cycles with sine and cosine.
- Domain knowledge gives flags, ratios and bends.
- Permutation importance: the score drop when a feature is shuffled.
- Partial dependence and what-if plots show how a feature moves predictions.

## What's Next

Next session is Project 3. You will finish the customer churn model: final features, model choice, one honest test score, explanations and a short report for the retention team.